# 1. Data Exploration — SQuAD Dataset

Before fine-tuning anything, let's actually look at the data we're working with.

**SQuAD (Stanford Question Answering Dataset)** is a reading comprehension dataset made up of questions posed by crowdworkers on a set of Wikipedia articles, where the answer to every question is a span of text from the corresponding passage.

In this notebook we'll:
- Download SQuAD v1.1 (using `requests`, no heavy dataset library needed just to look at the data)
- Parse it into a flat, easy-to-read `pandas` DataFrame
- Explore basic stats (number of examples, context/question/answer lengths)
- Save a small sample locally so later notebooks can load quickly

No GPU needed here at all — this is pure data wrangling.

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
import requests

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

TRAIN_URL = "https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v1.1.json"
DEV_URL = "https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v1.1.json"

TRAIN_PATH = DATA_DIR / "train-v1.1.json"
DEV_PATH = DATA_DIR / "dev-v1.1.json"


## Download the raw JSON files

Using plain `requests` here rather than a heavier dataset-loading library — SQuAD is
distributed as two JSON files (train/dev), so a simple GET request is all we need.
Files are cached to disk so we don't re-download every time we run this notebook.

In [2]:
def download_if_missing(url: str, dest: Path) -> None:
    if dest.exists():
        print(f"Already downloaded: {dest} ({dest.stat().st_size / 1e6:.1f} MB)")
        return
    print(f"Downloading {url} ...")
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    dest.write_bytes(response.content)
    print(f"Saved to {dest} ({dest.stat().st_size / 1e6:.1f} MB)")


download_if_missing(TRAIN_URL, TRAIN_PATH)
download_if_missing(DEV_URL, DEV_PATH)

Saved to ..\data\train-v1.1.json (30.3 MB)
Saved to ..\data\dev-v1.1.json (4.9 MB)


## Parse the nested JSON into a flat table

SQuAD's raw format is nested: `data -> paragraphs -> qas -> answers`. That's awkward to
explore directly, so let's flatten it into one row per (context, question, answer).

In [3]:
def squad_json_to_dataframe(path: Path) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        squad_dict = json.load(f)

    rows = []
    for article in squad_dict["data"]:
        for paragraph in article["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                question = qa["question"]
                qid = qa["id"]
                # Training set: use the first answer. Some questions have multiple
                # acceptable answers (esp. in the dev set), we just take the first here.
                if qa.get("answers"):
                    answer_text = qa["answers"][0]["text"]
                    answer_start = qa["answers"][0]["answer_start"]
                else:
                    answer_text, answer_start = None, None

                rows.append(
                    {
                        "id": qid,
                        "context": context,
                        "question": question,
                        "answer_text": answer_text,
                        "answer_start": answer_start,
                    }
                )
    return pd.DataFrame(rows)


train_df = squad_json_to_dataframe(TRAIN_PATH)
dev_df = squad_json_to_dataframe(DEV_PATH)

print(f"Train rows: {len(train_df):,}")
print(f"Dev rows:   {len(dev_df):,}")
train_df.head()

Train rows: 87,599
Dev rows:   10,570


,id,context,question,answer_text,answer_start
0,5733be284776f41900661182,"Architecturally, the school has a Catholic cha...",To whom did the Virgin Mary allegedly appear i...,Saint Bernadette Soubirous,515
1,5733be284776f4190066117f,"Architecturally, the school has a Catholic cha...",What is in front of the Notre Dame Main Building?,a copper statue of Christ,188
2,5733be284776f41900661180,"Architecturally, the school has a Catholic cha...",The Basilica of the Sacred heart at Notre Dame...,the Main Building,279
3,5733be284776f41900661181,"Architecturally, the school has a Catholic cha...",What is the Grotto at Notre Dame?,a Marian place of prayer and reflection,381
4,5733be284776f4190066117e,"Architecturally, the school has a Catholic cha...",What sits on top of the Main Building at Notre...,a golden statue of the Virgin Mary,92


## Quick sanity check — do the answer spans line up?

In [4]:
sample = train_df.iloc[0]
start = sample["answer_start"]
end = start + len(sample["answer_text"])

print("Question:", sample["question"])
print("Answer (from field):", sample["answer_text"])
print("Answer (sliced from context using answer_start):", sample["context"][start:end])
assert sample["context"][start:end] == sample["answer_text"], "Answer span mismatch!"
print("\nSpans line up correctly.")

Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Answer (from field): Saint Bernadette Soubirous
Answer (sliced from context using answer_start): Saint Bernadette Soubirous

Spans line up correctly.


## Basic stats

In [5]:
train_df["context_len_words"] = train_df["context"].str.split().apply(len)
train_df["question_len_words"] = train_df["question"].str.split().apply(len)
train_df["answer_len_words"] = train_df["answer_text"].str.split().apply(len)

train_df[["context_len_words", "question_len_words", "answer_len_words"]].describe()

,context_len_words,question_len_words,answer_len_words
count,87599.000000,87599.000000,87599.000000
mean,119.763125,10.061108,3.162159
std,49.365000,3.559230,3.392334
min,20.000000,1.000000,1.000000
25%,89.000000,8.000000,1.000000
50%,110.000000,10.000000,2.000000
75%,142.000000,12.000000,3.000000
max,653.000000,40.000000,43.000000


In [6]:
print(f"Unique contexts in train set: {train_df['context'].nunique():,}")
print(f"Average question length:  {train_df['question_len_words'].mean():.1f} words")
print(f"Average answer length:    {train_df['answer_len_words'].mean():.1f} words")
print(f"Most answers are short spans (median {train_df['answer_len_words'].median():.0f} words) — "
      f"this confirms SQuAD is extractive QA, not summarization.")

Unique contexts in train set: 18,891
Average question length:  10.1 words
Average answer length:    3.2 words
Most answers are short spans (median 2 words) — this confirms SQuAD is extractive QA, not summarization.


## Save a lightweight sample for quick iteration

Instead of keeping the full ~87k-row train set around, save a smaller CSV sample.
This keeps disk usage low and makes it fast to reload in the next notebook while
prototyping the fine-tuning loop.

In [7]:
SAMPLE_SIZE = 5000  # plenty for exploration; the fine-tuning notebook takes its own subset

sample_df = train_df.sample(n=min(SAMPLE_SIZE, len(train_df)), random_state=42)
sample_path = DATA_DIR / "train_sample.csv"
sample_df.to_csv(sample_path, index=False)

print(f"Saved {len(sample_df):,} rows to {sample_path} "
      f"({sample_path.stat().st_size / 1e6:.2f} MB on disk)")

Saved 5,000 rows to ..\data\train_sample.csv (4.36 MB on disk)


## Next step

Head over to `2_Fine_Tune_BERT_SQuAD.ipynb` to actually fine-tune a transformer on this data.